# 2. Data Understanding: Eksplorasi & Ekstraksi Fitur Klasik Citra Kerapian Alas Kaki

Tahap **Data Understanding** pada metodologi **CRISP-DM** berfokus pada memahami karakteristik dataset visual citra alas kaki (sandal dan sepatu) yang telah dikumpulkan.

### Sasaran Tahap Ini:
1. **Verifikasi Struktur Direktori:** Memastikan dataset terbagi ke dalam 2 kelas:
   * `RAPI` (Alas kaki tertata sejajar, teratur, atau berada di kompartemen rak).
   * `BERANTAKAN` (Alas kaki berserakan, bersilangan, bertumpuk, atau di luar batas teratur).
2. **Pemeriksaan Integritas & Dimensi:** Menghitung jumlah sampel citra pada 14 subdirektori kelompok observasi, memeriksa keberagaman resolusi, dan rasio aspek.
3. **Visualisasi Sampel Citra:** Menampilkan representasi visual sampel dari masing-masing kategori dan kelompok.
4. **Ekstraksi Fitur Warna Klasik (*Classical Color Feature Extraction*):**
   * Menghitung nilai tendensi sentral (**Rata-rata / Mean**) intensitas warna pada setiap kanal (**Red, Green, Blue**).
   * Menghitung dispersi sebaran piksel (**Standar Deviasi / Std**) pada setiap kanal (**Red, Green, Blue**).
   * Menyimpan hasil ekstraksi fitur tabular ke berkas `fitur_warna_klasik.csv`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageStat


def find_project_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / Path("src/1-project/proyek_sain_data/src/10-project_citra")):
            if (candidate / "3-data_preparation").is_dir() and (candidate / "6-deployment").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Folder proyek 10-project_citra tidak ditemukan dari direktori kerja.")


PROJECT_DIR = find_project_dir()
DATASET_DIR = PROJECT_DIR.parents[1] / "data"
CLASSES = ["RAPI", "BERANTAKAN"]
if not all((DATASET_DIR / cls).is_dir() for cls in CLASSES):
    raise FileNotFoundError(f"Folder kelas dataset tidak lengkap: {DATASET_DIR}")

for cls in CLASSES:
    files = [f for f in (DATASET_DIR / cls).rglob("*") if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    print(f"Kelas: {cls:<12} -> {len(files)} berkas gambar")
print(f"Total Citra Valid: {sum(len([f for f in (DATASET_DIR / cls).rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]) for cls in CLASSES)} berkas")

---
## 2.1 Analisis Distribusi Kelas & Inventarisasi Resolusi Citra

Dataset terdiri dari **153 citra** yang dikumpulkan dari 7 kelompok observasi lapangan dengan komposisi seimbang:
- **Kelas Rapi:** 73 sampel (47.7%)
- **Kelas Berantakan:** 80 sampel (52.3%)

Distribusi ini tergolong **sangat seimbang (*well-balanced*)**.


In [ ]:
manifest = []
for cls in CLASSES:
    folder = DATASET_DIR / cls
    for image_path in sorted(folder.rglob("*")):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            with Image.open(image_path) as img:
                width, height = img.size
                manifest.append({
                    "filename": image_path.name,
                    "rel_path": image_path.relative_to(DATASET_DIR).as_posix(),
                    "class": "Rapi" if cls == "RAPI" else "Berantakan",
                    "kelompok": image_path.parent.name,
                    "width": width,
                    "height": height,
                    "aspect_ratio": round(width / height, 2),
                    "format": img.format,
                    "mode": img.mode,
                })

df_info = pd.DataFrame(manifest)
print(f"Total citra terdata: {len(df_info)}")
print()
print("Ringkasan Resolusi & Aspek Rasio Citra:")
display(df_info[["width", "height", "aspect_ratio"]].describe())

---
## 2.2 Visualisasi Sampel Citra per Kategori

Berikut adalah galeri visualisasi representasi sampel dari kategori **Rapi** dan **Berantakan**:


In [ ]:
def plot_samples(dataset_dir, classes, n_cols=4):
    fig, axes = plt.subplots(len(classes), n_cols, figsize=(16, 4 * len(classes)))
    for row_idx, cls in enumerate(classes):
        cls_folder = dataset_dir / cls
        img_files = sorted([f for f in cls_folder.rglob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])[:n_cols]
        for col_idx, f in enumerate(img_files):
            ax = axes[row_idx, col_idx]
            with Image.open(f) as img:
                ax.imshow(img.convert('RGB'))
                label_name = 'Rapi' if cls == 'RAPI' else 'Berantakan'
                ax.set_title(f"{label_name} ({f.parent.name})\n{img.width}x{img.height}", fontsize=10)
                ax.axis('off')
    plt.suptitle("Sampel Citra Kategori Rapi vs Berantakan", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_samples(DATASET_DIR, CLASSES)


---
## 2.3 Ekstraksi Fitur Warna Klasik (Mean & Standar Deviasi Kanal R, G, B)

Sesuai metodologi CRISP-DM, untuk setiap citra dalam dataset diekstraksi fitur statistik warna dasar pada setiap kanal warnanya:

$$\mu_c = \frac{1}{N} \sum_{i=1}^{N} P_{c}(i) \quad \text{dan} \quad \sigma_c = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (P_{c}(i) - \mu_c)^2}$$

di mana $c \in \{\text{Red}, \text{Green}, \text{Blue}\}$ dan $N = W \times H$.


In [ ]:
def extract_classical_color_features(dataset_dir, classes):
    records = []
    for cls_name in classes:
        folder = dataset_dir / cls_name
        files = sorted([f for f in folder.rglob("*") if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        for image_path in files:
            with Image.open(image_path) as img:
                image_rgb = img.convert("RGB")
                width, height = image_rgb.size
                stats = ImageStat.Stat(image_rgb)
                red_mean, green_mean, blue_mean = stats.mean
                red_std, green_std, blue_std = stats.stddev
                class_name = "Rapi" if cls_name == "RAPI" else "Berantakan"
                records.append({
                    "filename": image_path.name,
                    "rel_path": image_path.relative_to(dataset_dir).as_posix(),
                    "class": class_name,
                    "label": int(class_name == "Rapi"),
                    "width": width,
                    "height": height,
                    "mean_red": round(red_mean, 4),
                    "std_red": round(red_std, 4),
                    "mean_green": round(green_mean, 4),
                    "std_green": round(green_std, 4),
                    "mean_blue": round(blue_mean, 4),
                    "std_blue": round(blue_std, 4),
                    "mean_overall": round((red_mean + green_mean + blue_mean) / 3, 4),
                    "std_overall": round((red_std + green_std + blue_std) / 3, 4),
                })
    return pd.DataFrame(records)


df_features = extract_classical_color_features(DATASET_DIR, CLASSES)
output_path = PROJECT_DIR / "2-data_understanding" / "fitur_warna_klasik.csv"
df_features.to_csv(output_path, index=False)
print(f"Ekstraksi selesai: {len(df_features)} baris disimpan di {output_path}")
display(df_features.head(10))

---
## 2.4 Analisis Perbandingan Fitur Warna Rata-rata & Standar Deviasi Antar-Kelas


In [ ]:
summary = df_features.groupby('class')[['mean_red', 'std_red', 'mean_green', 'std_green', 'mean_blue', 'std_blue', 'mean_overall', 'std_overall']].mean()
print("=== Rata-rata Fitur Warna Berdasarkan Kelas ===")
display(summary.round(3))


---
## 2.5 Kesimpulan Tahap Data Understanding

1. **Integritas Dataset:** Seluruh 153 citra dari 14 subdirektori kelompok lapangan berhasil dimuat dan divalidasi.
2. **Keseimbangan Kelas:** Kelas terbagi berimbang (73 Rapi vs 80 Berantakan).
3. **Fitur Tabular:** Data warna statistik telah disimpan ke [`fitur_warna_klasik.csv`](fitur_warna_klasik.csv) dan siap diproses ke modul Data Preparation.
